# Realized MOVE Index

In [1]:
import numpy as np
import pandas as pd
import polars as pl
import pandas_datareader.data as web
import yfinance as yf
import plotly.graph_objects as go

START   = "2000-01-01"
WINDOW  = 21
LAMBDA  = 0.94
ANNUAL  = np.sqrt(252)
WEIGHTS = {"y2": 0.20, "y5": 0.20, "y10": 0.40, "y30": 0.20}
FRED    = {"y2": "DGS2", "y5": "DGS5", "y10": "DGS10", "y30": "DGS30"}
TENORS  = list(WEIGHTS)

In [2]:
raw = web.DataReader(list(FRED.values()), "fred", START)
raw.columns = TENORS
raw.index.name = "date"
yields = (
    pl.from_pandas(raw.reset_index())
      .with_columns(pl.col("date").cast(pl.Date))
      .drop_nulls()
)
yields.tail()

date,y2,y5,y10,y30
date,f64,f64,f64,f64
2026-05-11,3.95,4.07,4.42,4.98
2026-05-12,4.0,4.12,4.46,5.03
2026-05-13,3.98,4.12,4.46,5.03
2026-05-14,4.0,4.13,4.47,5.02
2026-05-15,4.09,4.26,4.59,5.12


In [3]:
yields = yields.with_columns([
    ((pl.col(t) - pl.col(t).shift(1)) * 100).alias(f"d{t}")
    for t in TENORS
])
yields.select(["date", *(f"d{t}" for t in TENORS)]).tail()

date,dy2,dy5,dy10,dy30
date,f64,f64,f64,f64
2026-05-11,5.0,5.0,4.0,3.0
2026-05-12,5.0,5.0,4.0,5.0
2026-05-13,-2.0,0.0,0.0,0.0
2026-05-14,2.0,1.0,1.0,-1.0
2026-05-15,9.0,13.0,12.0,10.0


In [4]:
yields = yields.with_columns([
    ((pl.col(f"d{t}") ** 2).rolling_mean(window_size=WINDOW).sqrt() * ANNUAL).alias(f"rv_{t}")
    for t in TENORS
])
yields = yields.with_columns(
    sum(WEIGHTS[t] * pl.col(f"rv_{t}") for t in TENORS).alias("rMOVE_rolling")
)
yields.select(["date", *(f"rv_{t}" for t in TENORS), "rMOVE_rolling"]).tail()

date,rv_y2,rv_y5,rv_y10,rv_y30,rMOVE_rolling
date,f64,f64,f64,f64,f64
2026-05-11,69.627581,70.569115,60.893349,44.766059,61.349891
2026-05-12,70.992957,72.332565,62.353829,47.874837,63.181604
2026-05-13,70.992957,70.228199,60.794737,46.733286,61.908783
2026-05-14,71.330218,69.541355,60.0,46.346521,61.443619
2026-05-15,77.537088,82.776808,72.249567,56.178288,72.198264


In [5]:
yields = yields.with_columns([
    ((pl.col(f"d{t}") ** 2).ewm_mean(alpha=1 - LAMBDA, adjust=False).sqrt() * ANNUAL).alias(f"rv_ewma_{t}")
    for t in TENORS
])
yields = yields.with_columns(
    sum(WEIGHTS[t] * pl.col(f"rv_ewma_{t}") for t in TENORS).alias("rMOVE_ewma")
)
yields.select(["date", "rMOVE_rolling", "rMOVE_ewma"]).tail()

date,rMOVE_rolling,rMOVE_ewma
date,f64,f64
2026-05-11,61.349891,66.962819
2026-05-12,63.181604,67.414235
2026-05-13,61.908783,65.444498
2026-05-14,61.443619,63.636297
2026-05-15,72.198264,75.720902


In [6]:
move_raw = yf.download("^MOVE", start=START, auto_adjust=True, progress=False)
move_series = move_raw["Close"].squeeze().rename("MOVE")
move = pl.from_pandas(move_series.to_frame().reset_index()).rename({"Date": "date"})
move = move.with_columns(pl.col("date").cast(pl.Date))

realized_fwd = yields.select([
    "date",
    pl.col("rMOVE_rolling").shift(-WINDOW),
    pl.col("rMOVE_ewma").shift(-WINDOW),
])
final = realized_fwd.join(move, on="date", how="inner").drop_nulls()
final.tail()

date,rMOVE_rolling,rMOVE_ewma,MOVE
date,f64,f64,f64
2026-04-10,61.349891,66.962819,72.150002
2026-04-13,63.181604,67.414235,74.419998
2026-04-14,61.908783,65.444498,74.349998
2026-04-15,61.443619,63.636297,67.940002
2026-04-16,72.198264,75.720902,65.889999


In [7]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=final["date"], y=final["MOVE"],          name="MOVE (implied)",   mode="lines"))
fig.add_trace(go.Scatter(x=final["date"], y=final["rMOVE_rolling"], name="rMOVE 21d rolling", mode="lines"))
fig.add_trace(go.Scatter(x=final["date"], y=final["rMOVE_ewma"],    name="rMOVE EWMA λ=0.94", mode="lines"))
fig.update_layout(
    title="Implied vs Realized MOVE",
    yaxis_title="bp/year (annualized)",
    template="plotly_dark",
    yaxis=dict(showgrid=False),
    hovermode="x unified",
)
fig.show()

In [8]:
final = final.with_columns((pl.col("MOVE") - pl.col("rMOVE_rolling")).alias("vrp"))
vrp = final["vrp"]

print(f"mean   {vrp.mean():7.2f} bp")
print(f"median {vrp.median():7.2f} bp")
print(f"std    {vrp.std():7.2f} bp")
print(f" 5%    {vrp.quantile(0.05):7.2f} bp")
print(f"95%    {vrp.quantile(0.95):7.2f} bp")

fig_h = go.Figure(go.Histogram(x=vrp, nbinsx=80))
fig_h.update_layout(title="VRP distribution (MOVE − rMOVE_rolling)", xaxis_title="bp/year",
                    template="plotly_dark", yaxis=dict(showgrid=False))
fig_h.show()

fig_t = go.Figure(go.Scatter(x=final["date"], y=vrp, mode="lines", name="VRP"))
fig_t.add_hline(y=0, line_dash="dot", line_color="white")
fig_t.update_layout(title="Rates VRP time series", yaxis_title="bp/year",
                    template="plotly_dark", yaxis=dict(showgrid=False))
fig_t.show()

mean      5.66 bp
median    7.05 bp
std      21.06 bp
 5%     -25.96 bp
95%      35.57 bp


In [9]:
corr_rolling = float(np.corrcoef(final["MOVE"], final["rMOVE_rolling"])[0, 1])
corr_ewma    = float(np.corrcoef(final["MOVE"], final["rMOVE_ewma"])[0, 1])

n_move    = move.shape[0]
n_rolling = yields.filter(pl.col("rMOVE_rolling").is_not_null()).shape[0]
n_ewma    = yields.filter(pl.col("rMOVE_ewma").is_not_null()).shape[0]

checks = pd.DataFrame({
    "series":         ["MOVE", "rMOVE_rolling", "rMOVE_ewma"],
    "mean_joined":    [final["MOVE"].mean(), final["rMOVE_rolling"].mean(), final["rMOVE_ewma"].mean()],
    "n_days_defined": [n_move, n_rolling, n_ewma],
    "corr_with_MOVE": [1.0, corr_rolling, corr_ewma],
})
print(checks.to_string(index=False))

       series  mean_joined  n_days_defined  corr_with_MOVE
         MOVE    87.105980            5813        1.000000
rMOVE_rolling    81.450845            6575        0.774084
   rMOVE_ewma    82.296893            6595        0.838337
